# Experiment: Data Cleaning with Python (pandas)

## Aim
Take a raw, messy dataset and turn it into an analysis-ready table by detecting and repairing the common data-quality problems: untidy headers, disguised missing values, inconsistent labels, wrong data types, mixed date formats, duplicates, invalid values, outliers and missing data.

## Dataset
| | |
|---|---|
| **File** | `retail_orders_dirty.csv` |
| **Size** | 525 rows × 11 columns |
| **Content** | Synthetic retail orders: order id, customer, gender, age, city, e-mail, order date, product category, quantity, unit price, total |
| **Nature** | Data-quality problems were injected on purpose so that each cleaning technique has something to find |

## Requirements
Python 3.9+, `pandas`, `numpy` (`pip install pandas numpy`).

## How to run
Keep `retail_orders_dirty.csv` in the same folder as this notebook, then choose **Run All**.
In Google Colab, first upload the CSV through the **Files** panel on the left.

## Output
`retail_orders_clean.csv`, a cleaning log, and a before/after quality report.

## Step 0: Setup
Import the libraries and create a small log so that every cleaning action is counted and can be reported at the end.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

DATA_PATH = "retail_orders_dirty.csv"
OUT_PATH = "retail_orders_clean.csv"

log = []  # every cleaning action is recorded here: (step, action, cells/rows affected)


def record(step, action, n):
    log.append({"step": step, "action": action, "affected": int(n)})
    print(f"  -> {action}: {int(n)}")

## Step 1: Load and profile the raw data
Read every column as text and switch off automatic missing-value detection. Nothing should be converted silently before it has been inspected.

In [2]:
# Read every column as text and switch off automatic NA detection, so nothing
# is silently converted before we have looked at it.
raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
df = raw.copy()

print("Shape:", df.shape)
print("\nColumn names exactly as stored:", list(df.columns))
print("\nFirst rows:\n", df.head(8).to_string())
print("\nDistinct values per column:\n", df.nunique().to_string())

Shape: (525, 11)

Column names exactly as stored: ['Order ID', ' Customer Name', 'Gender', 'AGE', 'City ', 'E-mail', 'Order_Date', 'Product Category', 'Qty', 'Unit Price (KES)', 'Total']

First rows:
     Order ID   Customer Name  Gender AGE    City                         E-mail    Order_Date Product Category Qty Unit Price (KES)  Total
0   ORD-1395  Faith Kiplagat  Female  18  Nairobi  faith.kiplagat84@example.com    2025-12-20        Groceries   1             1770   1770
1   ORD-1260     Ruth Otieno  Female  39  Mombasa     ruth.otieno40@example.com    29/06/2025        Groceries   3             1290   3870
2   ORD-1162    Brian Chebet       M  18   Nakuru    brian.chebet19@example.com    30/08/2025            books   3             2000   6000
3   ORD-1101  Moses Nyambura    MALE  53  Eldoret  moses.nyambura65@example.com    2025-12-17            Books   3             2070   6210
4   ORD-1361     Brian Kamau   Male   47  Nairobi     brian.kamau41@example.com    2025-07-21     Home&K

## Step 2: Tidy the column names
Headers contain stray spaces, mixed case, punctuation and units. Convert them to one convention: lower-case `snake_case`.

In [3]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"\(.*?\)", "", regex=True)   # drop units such as "(KES)"
    .str.strip()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
df = df.rename(columns={"e_mail": "email", "qty": "quantity"})
print("Clean names:", list(df.columns))

TEXT_COLS = ["order_id", "customer_name", "gender", "city", "email", "product_category"]

Clean names: ['order_id', 'customer_name', 'gender', 'age', 'city', 'email', 'order_date', 'product_category', 'quantity', 'unit_price', 'total']


## Step 3: Strip whitespace and expose disguised missing values
Missing data is often hidden behind placeholders such as `?`, `N/A`, `-` or `unknown`. Until these become real `NaN` values, pandas counts them as valid data.

In [4]:
before = df.copy()
df = df.apply(lambda s: s.str.strip())
record(3, "cells with leading/trailing whitespace trimmed", (before != df).sum().sum())

# Placeholders that mean "missing" but are stored as ordinary text.
MISSING_TOKENS = {"", "n/a", "na", "null", "none", "nan", "?", "-", "unknown"}
is_missing = df.apply(lambda s: s.str.lower().isin(MISSING_TOKENS))
print("\nPlaceholders found:")
print(df.where(is_missing).stack().dropna().replace("", "(blank)").value_counts().to_string())
df = df.mask(is_missing)  # -> real NaN
record(3, "placeholder cells converted to NaN", is_missing.sum().sum())

missing_raw = df.isna().sum()
print("\nMissing values per column (after exposing placeholders):")
print(pd.DataFrame({"missing": missing_raw, "percent": (missing_raw / len(df) * 100).round(1)}).to_string())

  -> cells with leading/trailing whitespace trimmed: 152

Placeholders found:
(blank)    126
?           16
n/a         16
N/A         15
unknown     14
NULL        12
-           11
  -> placeholder cells converted to NaN: 210

Missing values per column (after exposing placeholders):
                  missing  percent
order_id                0      0.0
customer_name           0      0.0
gender                 27      5.1
age                    40      7.6
city                   30      5.7
email                  35      6.7
order_date             16      3.0
product_category       17      3.2
quantity                0      0.0
unit_price             20      3.8
total                  25      4.8


## Step 4: Standardise the key and remove duplicates
The same order can hide behind different spellings of its key (`ord-1001`, `ORD_1001`). Standardise the key first, then remove duplicates.

In [5]:
print("Exact duplicate rows BEFORE fixing the key:", df.duplicated().sum())

# The same order can hide behind different spellings of its id: "ord-1001", "ORD_1001"
df["order_id"] = df["order_id"].str.upper().str.replace(r"[\s_]+", "-", regex=True)
print("Exact duplicate rows AFTER fixing the key: ", df.duplicated().sum())

n = len(df)
df = df.drop_duplicates().reset_index(drop=True)
record(4, "duplicate rows removed", n - len(df))

# Any id still repeated would be a conflicting record that needs manual review.
conflicts = df[df["order_id"].duplicated(keep=False)]
print("Conflicting records sharing an order_id:", len(conflicts))

Exact duplicate rows BEFORE fixing the key: 22
Exact duplicate rows AFTER fixing the key:  25
  -> duplicate rows removed: 25
Conflicting records sharing an order_id: 0


## Step 5: Make categorical and text values consistent
One real-world category written many ways (`M`, `male`, `MALE`) splits a group into several. Map every variant to a single canonical label and assert that nothing is left unmapped.

In [6]:
print("gender BEFORE:", sorted(df["gender"].dropna().unique()))
df["gender"] = df["gender"].str.lower().map(
    {"m": "Male", "male": "Male", "f": "Female", "female": "Female"}
)
print("gender AFTER: ", sorted(df["gender"].dropna().unique()))

print("\ncity BEFORE:", sorted(df["city"].dropna().unique()))
city = df["city"].str.lower().str.replace(r"[^a-z ]", "", regex=True).str.strip()
city = city.replace({"nairobbi": "nairobi", "nbo": "nairobi", "mombassa": "mombasa"})
df["city"] = city.str.title()
print("city AFTER: ", sorted(df["city"].dropna().unique()))

print("\nproduct_category BEFORE:", sorted(df["product_category"].dropna().unique()))
cat = df["product_category"].str.lower().str.replace("&", " and ").str.replace(r"\s+", " ", regex=True)
cat = cat.replace({
    "electronic": "electronics", "clothes": "clothing", "grocery": "groceries",
    "book": "books", "home and kitchen": "home & kitchen",
})
df["product_category"] = cat.str.title()
print("product_category AFTER: ", sorted(df["product_category"].dropna().unique()))

# Guard: every label must now belong to the agreed vocabulary.
VALID = {
    "gender": {"Male", "Female"},
    "city": {"Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"},
    "product_category": {"Electronics", "Clothing", "Groceries", "Home & Kitchen", "Books"},
}
for col, allowed in VALID.items():
    unknown = set(df[col].dropna()) - allowed
    assert not unknown, f"Unmapped labels in {col}: {unknown}"

df["customer_name"] = df["customer_name"].str.replace(r"\s+", " ", regex=True).str.title()
df["email"] = df["email"].str.lower()
print("\nNames and e-mails normalised, e.g.:", df["customer_name"].head(3).tolist())

gender BEFORE: ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male']
gender AFTER:  ['Female', 'Male']

city BEFORE: ['ELDORET', 'Eldoret', 'Eldoret.', 'KISUMU', 'Kisumu', 'MOMBASA', 'Mombasa', 'Mombassa', 'NAIROBI', 'NBO', 'Nairobbi', 'Nairobi', 'Nakuru', 'eldoret', 'kisumu', 'mombasa', 'nairobi', 'nakuru']
city AFTER:  ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']

product_category BEFORE: ['BOOKS', 'Book', 'Books', 'Clothes', 'Clothing', 'ELECTRONICS', 'Electronic', 'Electronics', 'Groceries', 'Grocery', 'Home & Kitchen', 'Home and Kitchen', 'Home&Kitchen', 'books', 'clothing', 'electronics', 'groceries', 'home & kitchen']
product_category AFTER:  ['Books', 'Clothing', 'Electronics', 'Groceries', 'Home & Kitchen']

Names and e-mails normalised, e.g.: ['Faith Kiplagat', 'Ruth Otieno', 'Brian Chebet']


## Step 6: Convert each column to its proper data type
Convert each column from text to its proper type. Prices carry currency symbols and commas, some quantities are written as words, and dates arrive in four layouts. Each date layout is parsed explicitly, because guessing can silently swap day and month.

In [7]:
# 6a. Prices such as "KES 1,200", "Ksh1200", "1200.00"
price_txt = df["unit_price"]
df["unit_price"] = pd.to_numeric(price_txt.str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")
record(6, "unit_price values that needed symbol/comma removal",
       (price_txt.notna() & ~price_txt.str.fullmatch(r"\d+", na=False)).sum())

# 6b. Quantities typed as words
words = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}
record(6, "quantity values written as words", df["quantity"].str.lower().isin(words).sum())
df["quantity"] = pd.to_numeric(df["quantity"].str.lower().replace(words), errors="coerce").astype("Int64")

# 6c. Plain numeric columns ("Int64" is pandas' integer type that allows missing values)
df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")
df["total"] = pd.to_numeric(df["total"], errors="coerce")

# 6d. Dates arrive in four different layouts. Parse each layout explicitly;
#     guessing with a single call can silently swap day and month.
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y", "%B %d, %Y"]
date_txt = df["order_date"]
parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for fmt in DATE_FORMATS:
    attempt = pd.to_datetime(date_txt, format=fmt, errors="coerce")
    print(f"  format {fmt!r:14} matched {attempt.notna().sum():4d} rows")
    parsed = parsed.fillna(attempt)
bad_dates = date_txt[date_txt.notna() & parsed.isna()]
print("  Impossible calendar dates:", bad_dates.tolist())
record(6, "impossible dates set to missing", len(bad_dates))
df["order_date"] = parsed

print("\nData types now:\n", df.dtypes.to_string())

  -> unit_price values that needed symbol/comma removal: 83
  -> quantity values written as words: 15
  format '%Y-%m-%d'     matched  317 rows
  format '%d/%m/%Y'     matched   51 rows
  format '%d-%b-%Y'     matched   62 rows
  format '%B %d, %Y'    matched   53 rows
  Impossible calendar dates: ['31/04/2025', '31/04/2025']
  -> impossible dates set to missing: 2

Data types now:
 order_id                       str
customer_name                  str
gender                         str
age                          Int64
city                           str
email                          str
order_date          datetime64[ns]
product_category               str
quantity                     Int64
unit_price                 float64
total                      float64


## Step 7: Validate against domain rules
Check each column against its domain rule. A value that breaks a rule is wrong by definition, so it is set to missing and handled later with the other missing values.

In [8]:
# A value that breaks a rule is wrong by definition, so it is set to missing
# and handled later together with the other missing values.
rule = ~df["age"].between(18, 100) & df["age"].notna()
print("  ages outside 18-100:", sorted(df.loc[rule, "age"].tolist()))
df.loc[rule, "age"] = pd.NA
record(7, "invalid ages set to missing", rule.sum())

rule = (df["quantity"] < 1).fillna(False)
print("  quantities below 1:", sorted(df.loc[rule, "quantity"].tolist()))
df.loc[rule, "quantity"] = pd.NA
record(7, "invalid quantities set to missing", rule.sum())

# The extract covers orders placed during 2025 only.
rule = df["order_date"].notna() & ~df["order_date"].between("2025-01-01", "2025-12-31")
print("  dates outside 2025:", df.loc[rule, "order_date"].dt.date.tolist())
df.loc[rule, "order_date"] = pd.NaT
record(7, "out-of-period dates set to missing", rule.sum())

EMAIL_RE = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
rule = df["email"].notna() & ~df["email"].str.fullmatch(EMAIL_RE, na=False)
print("  malformed e-mails:", df.loc[rule, "email"].tolist())
df.loc[rule, "email"] = np.nan
record(7, "malformed e-mails set to missing", rule.sum())

  ages outside 18-100: [-5, -5, 3, 250]
  -> invalid ages set to missing: 4
  quantities below 1: [-2, -1, -1, -1, 0]
  -> invalid quantities set to missing: 5
  dates outside 2025: [datetime.date(2031, 6, 15)]
  -> out-of-period dates set to missing: 1
  malformed e-mails: ['moses.achieng59example.com', 'quentin.barasa6@example', 'yusuf.kamau38@', 'kevin.achieng10example.com', 'esther.otieno50@@example.com', 'yusuf.mutua6@example', 'esther.kamau98@@example.com', 'david.kiplagat42example.com']
  -> malformed e-mails set to missing: 8


## Step 8: Detect outliers
Flag extreme values with the interquartile-range rule (fences at Q1 − 1.5·IQR and Q3 + 1.5·IQR). Unit price is assessed separately inside each product category, because prices of different product types are not comparable.

In [9]:
def iqr_fences(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


# Quantity: one distribution for the whole table.
lo, hi = iqr_fences(df["quantity"].astype("float"))
qty_out = (df["quantity"] > hi).fillna(False)
print(f"  quantity fences: [{lo:.1f}, {hi:.1f}]  ->  outliers: {sorted(df.loc[qty_out, 'quantity'].tolist())}")

# Unit price: a phone and a loaf of bread are not comparable, so the fences
# are computed separately inside each product category.
fences = df.groupby("product_category")["unit_price"].apply(lambda s: pd.Series(iqr_fences(s), index=["lo", "hi"])).unstack()
print("\n  unit_price fences per category:\n", fences.round(0).to_string())
price_hi = df["product_category"].map(fences["hi"])
price_lo = df["product_category"].map(fences["lo"])
price_out = ((df["unit_price"] > price_hi) | (df["unit_price"] < price_lo)).fillna(False)
print("\n  unit_price outliers:\n", df.loc[price_out, ["order_id", "product_category", "quantity", "unit_price", "total"]].to_string())

# Flagged values are not deleted. They are set to missing so that Step 9 can
# try to recover the true value from the other columns of the same row.
df.loc[qty_out, "quantity"] = pd.NA
df.loc[price_out, "unit_price"] = np.nan
record(8, "quantity outliers flagged", qty_out.sum())
record(8, "unit_price outliers flagged", price_out.sum())

  quantity fences: [-1.0, 7.0]  ->  outliers: [250, 250, 250, 500, 500]

  unit_price fences per category:
                        lo        hi
product_category                   
Books             -1516.0    5254.0
Clothing          -1740.0    7980.0
Electronics      -35955.0  100785.0
Groceries         -1065.0    3695.0
Home & Kitchen    -8160.0   21440.0

  unit_price outliers:
      order_id product_category  quantity  unit_price     total
41   ORD-1390      Electronics         1    533000.0    5330.0
148  ORD-1490      Electronics         1   5515000.0   55150.0
305  ORD-1310         Clothing         3    508000.0   15240.0
478  ORD-1325      Electronics         5   3837000.0  191850.0
  -> quantity outliers flagged: 5
  -> unit_price outliers flagged: 4


## Step 9: Repair using redundancy:  total = quantity x unit_price
The three columns `quantity`, `unit_price` and `total` carry only two independent facts (`total = quantity × unit_price`). A single missing or suspect value can therefore be rebuilt exactly from the other two instead of being guessed.

In [10]:
# Three columns carry two independent facts, so one missing/suspect value can
# be rebuilt exactly from the other two instead of being guessed.
q = df["quantity"].astype("float")

implied_q = df["total"] / df["unit_price"]
ok = q.isna() & implied_q.notna() & (implied_q % 1 == 0) & implied_q.between(1, hi)
df.loc[ok, "quantity"] = implied_q[ok].astype(int)
record(9, "quantities recovered from total / unit_price", ok.sum())

implied_p = df["total"] / q
ok = df["unit_price"].isna() & implied_p.notna() & implied_p.between(price_lo, price_hi)
df.loc[ok, "unit_price"] = implied_p[ok]
record(9, "unit prices recovered from total / quantity", ok.sum())

# 'total' is a derived column: where it disagrees with its inputs, the inputs win.
expected = df["quantity"].astype("float") * df["unit_price"]
mismatch = df["total"].notna() & expected.notna() & ~np.isclose(df["total"], expected)
print("  examples of totals that disagree with quantity x unit_price:")
print(df.loc[mismatch, ["order_id", "quantity", "unit_price", "total"]].head(5).to_string())
record(9, "inconsistent totals found (recomputed in Step 10)", mismatch.sum())

  -> quantities recovered from total / unit_price: 8
  -> unit prices recovered from total / quantity: 22
  examples of totals that disagree with quantity x unit_price:
     order_id  quantity  unit_price    total
20   ORD-1244         1      1890.0   1390.0
26   ORD-1168         5      5670.0  29350.0
88   ORD-1478         1      9240.0  19239.0
126  ORD-1074         4      1570.0   7280.0
171  ORD-1096         2      3240.0  16479.0
  -> inconsistent totals found (recomputed in Step 10): 17


## Step 10: Handle the remaining missing values
Impute what is still missing. Numeric columns use the median of the most similar group and keep a flag column; categorical columns get an explicit `Unknown` level; identifiers and dates cannot be estimated honestly, so they stay missing.

In [11]:
print("Missing before imputation:\n", df.isna().sum().to_string(), "\n")

# Numeric columns: impute with a robust statistic from the most similar group,
# and keep a flag so that analysts can exclude imputed values if they wish.
df["age_imputed"] = df["age"].isna()
age_fill = df.groupby("gender")["age"].transform("median").fillna(df["age"].median())
df["age"] = df["age"].fillna(age_fill.round().astype("Int64"))
record(10, "ages imputed with the median of the same gender", df["age_imputed"].sum())

df["price_imputed"] = df["unit_price"].isna()
price_fill = df.groupby("product_category")["unit_price"].transform("median").fillna(df["unit_price"].median())
df["unit_price"] = df["unit_price"].fillna(price_fill)
record(10, "unit prices imputed with the median of the same category", df["price_imputed"].sum())

df["quantity_imputed"] = df["quantity"].isna()
df["quantity"] = df["quantity"].fillna(int(df["quantity"].median()))
record(10, "quantities imputed with the overall median", df["quantity_imputed"].sum())

# Categorical columns: an explicit "Unknown" level is safer than inventing a label.
for col in ["gender", "city", "product_category"]:
    record(10, f"{col} missing -> 'Unknown'", df[col].isna().sum())
    df[col] = df[col].fillna("Unknown")

# Identifiers and dates cannot be estimated honestly: they stay missing.
print("  email and order_date are left missing (cannot be inferred from other columns)")

# Derived column is rebuilt last, from the cleaned inputs.
df["total"] = df["quantity"].astype("float") * df["unit_price"]

Missing before imputation:
 order_id             0
customer_name        0
gender              25
age                 44
city                30
email               43
order_date          18
product_category    15
quantity             2
unit_price           2
total               25 

  -> ages imputed with the median of the same gender: 44
  -> unit prices imputed with the median of the same category: 2
  -> quantities imputed with the overall median: 2
  -> gender missing -> 'Unknown': 25
  -> city missing -> 'Unknown': 30
  -> product_category missing -> 'Unknown': 15
  email and order_date are left missing (cannot be inferred from other columns)


## Step 11: Final types, validation and export
Set the final data types, then express the quality rules as assertions. The notebook stops here if any rule is still broken.

In [12]:
df["unit_price"] = df["unit_price"].round(2)
df["total"] = df["total"].round(2)
for col in ["gender", "city", "product_category"]:
    df[col] = df[col].astype("category")
df = df.sort_values("order_id").reset_index(drop=True)

# Executable checks: the script stops here if any quality rule is still broken.
assert df["order_id"].is_unique, "duplicate order ids remain"
assert df["order_id"].str.fullmatch(r"ORD-\d{4}").all(), "malformed order id"
assert df[["order_id", "customer_name", "age", "quantity", "unit_price", "total"]].notna().all().all()
assert df["age"].between(18, 100).all()
assert (df["quantity"] >= 1).all()
assert np.isclose(df["total"], df["quantity"].astype("float") * df["unit_price"]).all()
assert df["order_date"].dropna().between("2025-01-01", "2025-12-31").all()
assert df["email"].dropna().str.fullmatch(EMAIL_RE).all()
print("All validation checks passed.")

df.to_csv(OUT_PATH, index=False)
print(f"Clean data saved to {OUT_PATH}")

All validation checks passed.
Clean data saved to retail_orders_clean.csv


## Results
The cleaning log, a before/after quality report, and a sample of the clean data.

In [13]:
print("Cleaning log")
print(pd.DataFrame(log).to_string(index=False))

raw_named = raw.copy()
raw_named.columns = [c for c in df.columns if not c.endswith("_imputed")]
cols = list(raw_named.columns)
report = pd.DataFrame({
    "dtype_before": "text",
    "dtype_after": df[cols].dtypes.astype(str),
    "missing_before": missing_raw[cols],       # after exposing placeholders (Step 3)
    "missing_after": df[cols].isna().sum(),
    "distinct_before": raw_named.nunique(),
    "distinct_after": df[cols].nunique(),
})
print("\nBefore / after quality report")
print(report.to_string())
print("(email and order_date gain missing values because invalid entries were blanked, not guessed)")

print(f"\nRows: {len(raw)} -> {len(df)}")
print("\nClean data sample:\n", df.head(8).to_string())
print("\nSummary statistics:\n", df[["age", "quantity", "unit_price", "total"]].describe().round(1).to_string())

Cleaning log
 step                                                   action  affected
    3           cells with leading/trailing whitespace trimmed       152
    3                       placeholder cells converted to NaN       210
    4                                   duplicate rows removed        25
    6       unit_price values that needed symbol/comma removal        83
    6                         quantity values written as words        15
    6                          impossible dates set to missing         2
    7                              invalid ages set to missing         4
    7                        invalid quantities set to missing         5
    7                       out-of-period dates set to missing         1
    7                         malformed e-mails set to missing         8
    8                                quantity outliers flagged         5
    8                              unit_price outliers flagged         4
    9             quantities recovered


Before / after quality report
                 dtype_before     dtype_after  missing_before  missing_after  distinct_before  distinct_after
order_id                 text             str               0              0              510             500
customer_name            text             str               0              0              313             268
gender                   text        category              27              0               16               3
age                      text           Int64              40              0               58              48
city                     text        category              30              0               28               6
email                    text             str              35             43              469             455
order_date               text  datetime64[ns]              16             18              366             270
product_category         text        category              17              0             

## Conclusion
The raw file of 525 rows was reduced to 500 unique, validated orders. Column names, labels, data types and date formats were standardised; placeholders, invalid values and outliers were exposed as missing; most suspect quantities and prices were recovered exactly from the redundant `total` column; the remainder were imputed and flagged.

**Points for discussion**
1. Why must placeholders be converted to `NaN` *before* counting missing values or removing duplicates?
2. Why is the outlier rule for `unit_price` applied per category rather than to the whole column?
3. Step 9 trusts `total` when `quantity` or `unit_price` is suspect. What happens when `total` itself is wrong, and how could you guard against it?
4. What are the risks of median imputation, and why are the `*_imputed` flag columns kept?
5. `email` and `order_date` end with more missing values than they started with. Why is that an improvement in data quality?